# Solution B2 · The price depends on where you are: KNN regressor on latitude and longitude

**Type**: Implement with scikit-learn

## Objective

Compare a linear regression with several KNN regressors on house prices in California, show that the KNN with **only latitude and longitude** beats the linear model with all the features, and
that a distance with **per-feature weights** (small weight on income) beats both even more. The weight is chosen on a validation set, not on the test set.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
GEO = ["Latitude", "Longitude"]


def rmse_of(model, columns):
    return mean_squared_error(y_te, model.fit(X_tr[columns], y_tr).predict(X_te[columns]), squared=False)

## Commented solution

In [ ]:
rmse = {
    "linear": rmse_of(LinearRegression(), list(X.columns)),
    "raw knn": rmse_of(KNeighborsRegressor(15), list(X.columns)),
    "scaled knn": rmse_of(make_pipeline(StandardScaler(), KNeighborsRegressor(15)), list(X.columns)),
    "geo knn": rmse_of(KNeighborsRegressor(15), GEO),
    "geo + income knn": rmse_of(make_pipeline(StandardScaler(), KNeighborsRegressor(15)), GEO + ["MedInc"]),
}
print(pd.Series(rmse).round(3).to_string())

In [ ]:
search = GridSearchCV(KNeighborsRegressor(), {"n_neighbors": [3, 5, 10, 15, 25, 40, 60], "weights": ["uniform", "distance"]}, cv=5, scoring="neg_root_mean_squared_error").fit(X_tr[GEO], y_tr)
geo_params = search.best_params_
best_geo_rmse = mean_squared_error(y_te, search.predict(X_te[GEO]), squared=False)
print(f"best parameters on the coordinates: {geo_params} | test RMSE {best_geo_rmse:.4f}")

In [ ]:
cols = GEO + ["MedInc"]
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=1)


def rmse_with_weight(w, X_a, y_a, X_b, y_b):
    sc = StandardScaler().fit(X_a[cols])
    A, B = sc.transform(X_a[cols]), sc.transform(X_b[cols])
    A[:, 2] *= w                                            # reduces the weight of the income in the distance
    B[:, 2] *= w
    return mean_squared_error(y_b, KNeighborsRegressor(10, weights="distance").fit(A, y_a).predict(B), squared=False)


weights = [0, 0.05, 0.1, 0.2, 0.35, 0.5, 1]
rmse_per_w = pd.Series({w: rmse_with_weight(w, X_fit, y_fit, X_val, y_val) for w in weights})
best_w = rmse_per_w.idxmin()
weighted_rmse = rmse_with_weight(best_w, X_tr, y_tr, X_te, y_te)
print(rmse_per_w.round(4).to_string())
print(f"best weight of the income: {best_w} | test RMSE with the weights: {weighted_rmse:.4f} (coordinates only: {best_geo_rmse:.4f}, linear: {rmse['linear']:.4f})")

## Check

In [ ]:
with verify():
    check(set(rmse) == {"linear", "raw knn", "scaled knn", "geo knn", "geo + income knn"}, "five models compared", "keys: linear, raw knn, scaled knn, geo knn, geo + income knn")
    check(rmse["raw knn"] > rmse["linear"] and rmse["scaled knn"] < rmse["linear"], "without scaling the KNN is worse than the linear one; with scaling better", "Population (scale 1000) dominates the raw distance")
    check(rmse["geo knn"] < rmse["scaled knn"] - 0.05 and rmse["geo knn"] < rmse["linear"] - 0.15, "the KNN with just two coordinates beats the one with 8 features and the linear one (by over 0.15)", "fewer features but the right ones")
    check(rmse["geo + income knn"] >= rmse["geo knn"] - 0.02, "adding the income with the same weight as the coordinates does not improve (or barely)", "the standardized income weighs as much as latitude and longitude")
    check(geo_params["n_neighbors"] <= 15 and best_geo_rmse < 0.55, "the search on the coordinates alone chooses a small k (15 or less); RMSE below 0.55", "GridSearchCV on Latitude and Longitude")
    check(list(rmse_per_w.index) == [0, 0.05, 0.1, 0.2, 0.35, 0.5, 1] and best_w == rmse_per_w.idxmin() and 0 < best_w <= 0.2, "the best weight of the income is small but nonzero (between 0.05 and 0.2)", "try the seven weights on the validation set")
    check(weighted_rmse < rmse["geo knn"] - 0.03 and weighted_rmse < best_geo_rmse, "with the weights, the test RMSE drops by over 0.03 compared to the coordinates alone and also beats the search on k", "retrain with best_w on the whole training set")
done()

## What you learned

- **Without scaling the KNN is worse than the linear one** (`Population` has a scale of thousands and alone decides the distances); with scaling it improves but stays below the KNN on the two coordinates alone: the other six features add noise to the
  distance (neighborhoods near in feature space are not near on the map).
- A KNN on latitude and longitude is a **geographic proximity model**: it beats the linear regression with all the features (0.53 against 0.74) because the price of houses depends on where you are in a nonlinear way (coast, metropolitan areas) and
  a linear model cannot represent it without engineered features.
- **The weights in the distance are hyperparameters.** Adding the income with weight 1 makes it worse, with a weight of about 0.05-0.1 it improves a lot (test RMSE 0.46 with `w = 0.05`): the information on income is useful, but only to break ties between geographic neighbors. It is the first step toward
  "learning the metric" (metric learning, NCA).
- The choice of `w` must be made on a validation set or in cross-validation, never on the test set: here you did it with 20% of the training set and the test set measured once.

## Common pitfall

Concluding "KNN is better than linear models" from this exercise. It holds here because the structure is spatial and the coordinates carry enormous weight; on other problems (many features, additive effects) the opposite holds (see C1 and F2).

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you decide which features to use in a KNN and with what weight?</summary>

I start from a few features that have a proximity meaning for the problem (coordinates, time, product categories), scale them and try a weight per feature (or group of features) as a hyperparameter, evaluating on a validation set or in cross-validation.
If the candidate features are many, I use a selection or a reduction (PCA, metric learning) before the KNN, or I switch to a model that learns by itself which features matter (trees, regularized linear models).

</details>